# BT4012 Kaggle 2026 — Illicit Bitcoin Transaction Detection

**Task:** predict the probability that each test transaction is illicit. **Metric:** ROC AUC.

**Stage 1 (this version): validation framework + baseline comparison.**
- Validation design **A1**: three expanding-window forward folds, because test time steps (36–49) all come after training (1–35).
- Model choice **B3**: compare Random Forest and default LightGBM on all 165 features; submit the one with the better mean fold AUC (minimum fold AUC as tie-break).

*AI use: Claude (Claude Code) was used to write boilerplate code for this notebook; validation and model choices were made by me.*

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

SEED = 42
DATA = Path('.')                 # folder holding train.csv, test.csv, txs_edgelist.csv, sample_submission.csv
SUB_DIR = Path('submissions'); SUB_DIR.mkdir(exist_ok=True)
LOG = Path('experiment_log.csv')
np.random.seed(SEED)

## 1. Load data

- `label` is blank for unknown transactions. Unknown is **not** licit, so those rows are excluded from supervised training and evaluation.
- Features are read as float32 to halve memory.

In [2]:
feat_cols = [f'feat_{i}' for i in range(1, 166)]
dtypes = {c: 'float32' for c in feat_cols}

train = pd.read_csv(DATA / 'train.csv', dtype=dtypes)
test = pd.read_csv(DATA / 'test.csv', dtype=dtypes)
sample = pd.read_csv(DATA / 'sample_submission.csv')

lab = train[train['label'].notna()].copy()
lab['label'] = lab['label'].astype(int)

print('train', train.shape, '| labelled', lab.shape, '| test', test.shape)
print('illicit rate (labelled):', round(lab['label'].mean(), 4))
print('train steps', train.time_step.min(), '-', train.time_step.max(), '| test steps', test.time_step.min(), '-', test.time_step.max())
assert (test['index'].values == sample['index'].values).all(), 'test order differs from sample submission'

train (141772, 168) | labelled (31235, 168) | test (15329, 168)
illicit rate (labelled): 0.1167
train steps 1 - 35 | test steps 36 - 49


## 2. Validation design (A1): forward-in-time folds

Each fold trains only on earlier steps and validates on later ones, mimicking the train → test gap.
`time_step` is used only for splitting, never as a model feature (test values 36–49 never occur in training).

In [3]:
FOLDS = {
    'F1': (range(1, 21), range(21, 28)),
    'F2': (range(1, 28), range(28, 36)),
    'F3': (range(1, 25), range(25, 36)),   # long horizon, closest to the 14-step test window
}
for name, (tr_steps, va_steps) in FOLDS.items():
    va = lab[lab.time_step.isin(va_steps)]
    print(f'{name}: train {len(lab[lab.time_step.isin(tr_steps)]):>6} rows | valid {len(va):>5} rows, {va.label.sum()} illicit')

F1: train  18889 rows | valid  6034 rows, 686 illicit
F2: train  24923 rows | valid  6312 rows, 1187 illicit
F3: train  23606 rows | valid  7629 rows, 1425 illicit


## 3. Candidate models (B3)

- **Random Forest:** 500 trees, `max_features='sqrt'`; averaging deep trees tends to be stable under drift.
- **LightGBM:** library defaults (only seed and verbosity set).

In [4]:
def make_model(name):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=500, max_features='sqrt', n_jobs=-1, random_state=SEED)
    if name == 'lgbm':
        return LGBMClassifier(random_state=SEED, verbose=-1)
    raise ValueError(name)

MODELS = ['rf', 'lgbm']

## 4. Cross-validation

In [5]:
rows = []
for m in MODELS:
    for fold, (tr_steps, va_steps) in FOLDS.items():
        tr = lab[lab.time_step.isin(tr_steps)]
        va = lab[lab.time_step.isin(va_steps)]
        t0 = time.time()
        model = make_model(m).fit(tr[feat_cols], tr['label'])
        auc = roc_auc_score(va['label'], model.predict_proba(va[feat_cols])[:, 1])
        rows.append({'model': m, 'fold': fold, 'auc': auc, 'sec': round(time.time() - t0, 1)})
        print(f'{m:5s} {fold}: AUC {auc:.4f}')

cv = pd.DataFrame(rows)
summary = cv.pivot(index='model', columns='fold', values='auc')
summary['mean'] = summary[list(FOLDS)].mean(axis=1)
summary['min'] = summary[list(FOLDS)].min(axis=1)
summary = summary.sort_values(['mean', 'min'], ascending=False)
summary.round(4)

rf    F1: AUC 0.8781


rf    F2: AUC 0.9953


rf    F3: AUC 0.9953


lgbm  F1: AUC 0.9451


lgbm  F2: AUC 0.9964


lgbm  F3: AUC 0.9965


fold,F1,F2,F3,mean,min
model,,,,,
lgbm,0.9451,0.9964,0.9965,0.9793,0.9451
rf,0.8781,0.9953,0.9953,0.9562,0.8781


## 5. Fit the winner on all labelled steps (1–35) and write the submission

Decision rule fixed before seeing results: highest mean fold AUC, minimum fold AUC as tie-break.

In [6]:
best = summary.index[0]
print('Selected model:', best)

final = make_model(best).fit(lab[feat_cols], lab['label'])
pred = final.predict_proba(test[feat_cols])[:, 1]

sub = sample[['index']].copy()
sub['target'] = pred

# schema checks against the template
assert sub.shape == sample.shape
assert list(sub.columns) == list(sample.columns)
assert (sub['index'].values == sample['index'].values).all()
assert np.isfinite(sub['target']).all() and sub['target'].between(0, 1).all()

sub_path = SUB_DIR / f'sub01_{best}.csv'
sub.to_csv(sub_path, index=False)
print('saved', sub_path)
sub.head()

Selected model: lgbm


saved submissions\sub01_lgbm.csv


,index,target
0,0,0.000453
1,1,0.000486
2,2,0.000441
3,3,0.000256
4,4,0.000074


## 6. Experiment log

In [7]:
log_row = pd.DataFrame([{
    'exp_id': 'exp01', 'date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
    'features': 'raw feat_1..feat_165', 'model': best,
    'params': str(make_model(best).get_params()), 'seed': SEED,
    **{f'auc_{f}': round(summary.loc[best, f], 4) for f in FOLDS},
    'cv_mean': round(summary.loc[best, 'mean'], 4), 'cv_min': round(summary.loc[best, 'min'], 4),
    'submission': str(sub_path), 'public_lb': '',
    'notes': 'Stage 1 baseline comparison (A1 folds, B3 rf vs lgbm)',
}])
log_row.to_csv(LOG, mode='a', header=not LOG.exists(), index=False)
pd.read_csv(LOG)

,exp_id,date,features,model,params,seed,auc_F1,auc_F2,auc_F3,cv_mean,cv_min,submission,public_lb,notes
0,exp01,2026-10-21 12:20,raw feat_1..feat_165,lgbm,"{'boosting_type': 'gbdt', 'class_weight': None...",42,0.9451,0.9964,0.9965,0.9793,0.9451,submissions\sub01_lgbm.csv,NaN,"Stage 1 baseline comparison (A1 folds, B3 rf v..."
